In [1]:
# PURPOSE OF NOTEBOOK

# Only verify:
# files readable
# schema correct
# data loaded successfully
# NO transformations here.

# imports

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,BooleanType,DateType,DecimalType
from pyspark.sql.functions import col,when,sum,avg,row_number
from pyspark.sql.window import Window

# Spark session

In [4]:

try:
    spark.stop()
except:
    pass
from pyspark.sql import SparkSession 

spark = (
    SparkSession.builder
    .appName("S3App")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

# # Enable S3A
# spark._jsc.hadoopConfiguration().set(
#     "fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem"
# )
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/26 06:58:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
pwd

'/opt/ipl_project_spark_setup/spark-notebooks'

In [6]:
import os

os.getcwd()

'/opt/ipl_project_spark_setup/spark-notebooks'

# Path variables

In [7]:
# Base path for raw data
BASE_PATH = "/opt/spark-data/1-raw"

# File paths
Ball_By_Ball_file = f"{BASE_PATH}/Ball_By_Ball.csv"
Match_file = f"{BASE_PATH}/Match.csv"
Player_file = f"{BASE_PATH}/Player.csv"
Player_match_file = f"{BASE_PATH}/Player_match.csv"
Team_file = f"{BASE_PATH}/Team.csv"

# Reading csv files to check columns

In [8]:

# data files reading ,this does not shuffle data so we can read and check columns 
# but (Infer Schema will use resources so avaoid for columnchecking on huge dATA) 

ball_by_ball_df = spark.read.csv(Ball_By_Ball_file, header=True, inferSchema=True)

match_df = spark.read.csv(Match_file, header=True, inferSchema=True)

player_df = spark.read.csv(Player_file, header=True, inferSchema=True)

player_match_df = spark.read.csv(Player_match_file, header=True, inferSchema=True)

team_df = spark.read.csv(Team_file, header=True, inferSchema=True)

In [9]:
print( ball_by_ball_df.columns)

['MatcH_id', 'Over_id', 'Ball_id', 'Innings_No', 'Team_Batting', 'Team_Bowling', 'Striker_Batting_Position', 'Extra_Type', 'Runs_Scored', 'Extra_runs', 'Wides', 'Legbyes', 'Byes', 'Noballs', 'Penalty', 'Bowler_Extras', 'Out_type', 'Caught', 'Bowled', 'Run_out', 'LBW', 'Retired_hurt', 'Stumped', 'caught_and_bowled', 'hit_wicket', 'ObstructingFeild', 'Bowler_Wicket', 'Match_Date', 'Season', 'Striker', 'Non_Striker', 'Bowler', 'Player_Out', 'Fielders', 'Striker_match_SK', 'StrikerSK', 'NonStriker_match_SK', 'NONStriker_SK', 'Fielder_match_SK', 'Fielder_SK', 'Bowler_match_SK', 'BOWLER_SK', 'PlayerOut_match_SK', 'BattingTeam_SK', 'BowlingTeam_SK', 'Keeper_Catch', 'Player_out_sk', 'MatchDateSK']


In [10]:
print( ball_by_ball_df.columns)
print(match_df.columns)
print(player_df.columns)
print(player_match_df.columns)
print(team_df.columns)

['MatcH_id', 'Over_id', 'Ball_id', 'Innings_No', 'Team_Batting', 'Team_Bowling', 'Striker_Batting_Position', 'Extra_Type', 'Runs_Scored', 'Extra_runs', 'Wides', 'Legbyes', 'Byes', 'Noballs', 'Penalty', 'Bowler_Extras', 'Out_type', 'Caught', 'Bowled', 'Run_out', 'LBW', 'Retired_hurt', 'Stumped', 'caught_and_bowled', 'hit_wicket', 'ObstructingFeild', 'Bowler_Wicket', 'Match_Date', 'Season', 'Striker', 'Non_Striker', 'Bowler', 'Player_Out', 'Fielders', 'Striker_match_SK', 'StrikerSK', 'NonStriker_match_SK', 'NONStriker_SK', 'Fielder_match_SK', 'Fielder_SK', 'Bowler_match_SK', 'BOWLER_SK', 'PlayerOut_match_SK', 'BattingTeam_SK', 'BowlingTeam_SK', 'Keeper_Catch', 'Player_out_sk', 'MatchDateSK']
['Match_SK', 'match_id', 'Team1', 'Team2', 'match_date', 'Season_Year', 'Venue_Name', 'City_Name', 'Country_Name', 'Toss_Winner', 'match_winner', 'Toss_Name', 'Win_Type', 'Outcome_Type', 'ManOfMach', 'Win_Margin', 'Country_id']
['PLAYER_SK', 'Player_Id', 'Player_Name', 'DOB', 'Batting_hand', 'Bowling

# checking default schema present in data ,
# But final schema will be after manual schema preparations and verification from data 
# and then new schema will be created for use

In [11]:
ball_by_ball_df.printSchema()
match_df.printSchema()
player_df.printSchema()
player_match_df.printSchema()
team_df.printSchema()



root
 |-- MatcH_id: integer (nullable = true)
 |-- Over_id: integer (nullable = true)
 |-- Ball_id: integer (nullable = true)
 |-- Innings_No: integer (nullable = true)
 |-- Team_Batting: string (nullable = true)
 |-- Team_Bowling: string (nullable = true)
 |-- Striker_Batting_Position: integer (nullable = true)
 |-- Extra_Type: string (nullable = true)
 |-- Runs_Scored: integer (nullable = true)
 |-- Extra_runs: integer (nullable = true)
 |-- Wides: integer (nullable = true)
 |-- Legbyes: integer (nullable = true)
 |-- Byes: integer (nullable = true)
 |-- Noballs: integer (nullable = true)
 |-- Penalty: integer (nullable = true)
 |-- Bowler_Extras: integer (nullable = true)
 |-- Out_type: string (nullable = true)
 |-- Caught: integer (nullable = true)
 |-- Bowled: integer (nullable = true)
 |-- Run_out: integer (nullable = true)
 |-- LBW: integer (nullable = true)
 |-- Retired_hurt: integer (nullable = true)
 |-- Stumped: integer (nullable = true)
 |-- caught_and_bowled: integer (null


# Section 4 — Schemas

In [12]:
ball_by_ball_schema = StructType([
    StructField("match_id", IntegerType(), True),
    StructField("over_id", IntegerType(), True),
    StructField("ball_id", IntegerType(), True),
    StructField("innings_no", IntegerType(), True),
    StructField("team_batting", StringType(), True),
    StructField("team_bowling", StringType(), True),
    StructField("striker_batting_position", IntegerType(), True),
    StructField("extra_type", StringType(), True),
    StructField("runs_scored", IntegerType(), True),
    StructField("extra_runs", IntegerType(), True),
    StructField("wides", IntegerType(), True),
    StructField("legbyes", IntegerType(), True),
    StructField("byes", IntegerType(), True),
    StructField("noballs", IntegerType(), True),
    StructField("penalty", IntegerType(), True),
    StructField("bowler_extras", IntegerType(), True),
    StructField("out_type", StringType(), True),
    StructField("caught", BooleanType(), True),
    StructField("bowled", BooleanType(), True),
    StructField("run_out", BooleanType(), True),
    StructField("lbw", BooleanType(), True),
    StructField("retired_hurt", BooleanType(), True),
    StructField("stumped", BooleanType(), True),
    StructField("caught_and_bowled", BooleanType(), True),
    StructField("hit_wicket", BooleanType(), True),
    StructField("obstructingfeild", BooleanType(), True),
    StructField("bowler_wicket", BooleanType(), True),
    StructField("match_date", DateType(), True),
    StructField("season", IntegerType(), True),
    StructField("striker", IntegerType(), True),
    StructField("non_striker", IntegerType(), True),
    StructField("bowler", IntegerType(), True),
    StructField("player_out", IntegerType(), True),
    StructField("fielders", IntegerType(), True),
    StructField("striker_match_sk", IntegerType(), True),
    StructField("strikersk", IntegerType(), True),
    StructField("nonstriker_match_sk", IntegerType(), True),
    StructField("nonstriker_sk", IntegerType(), True),
    StructField("fielder_match_sk", IntegerType(), True),
    StructField("fielder_sk", IntegerType(), True),
    StructField("bowler_match_sk", IntegerType(), True),
    StructField("bowler_sk", IntegerType(), True),
    StructField("playerout_match_sk", IntegerType(), True),
    StructField("battingteam_sk", IntegerType(), True),
    StructField("bowlingteam_sk", IntegerType(), True),
    StructField("keeper_catch", BooleanType(), True),
    StructField("player_out_sk", IntegerType(), True),
    StructField("matchdatesk", DateType(), True)
])



In [13]:
# creating schema variable manually
match_schema = StructType([
    StructField("match_sk", IntegerType(), True),
    StructField("match_id", IntegerType(), True),
    StructField("team1", StringType(), True),
    StructField("team2", StringType(), True),
    StructField("match_date", DateType(), True),
    StructField("season_year", IntegerType(), True),
    StructField("venue_name", StringType(), True),
    StructField("city_name", StringType(), True),
    StructField("country_name", StringType(), True),
    StructField("toss_winner", StringType(), True),
    StructField("match_winner", StringType(), True),
    StructField("toss_name", StringType(), True),
    StructField("win_type", StringType(), True),
    StructField("outcome_type", StringType(), True),
    StructField("manofmach", StringType(), True),
    StructField("win_margin", IntegerType(), True),
    StructField("country_id", IntegerType(), True)
])


In [14]:
# creating schema variable manually
player_schema = StructType([
    StructField("player_sk", IntegerType(), True),
    StructField("player_id", IntegerType(), True),
    StructField("player_name", StringType(), True),
    StructField("dob", DateType(), True),
    StructField("batting_hand", StringType(), True),
    StructField("bowling_skill", StringType(), True),
    StructField("country_name", StringType(), True)
])




In [15]:

# creating schema variable manually
player_match_schema = StructType([
    StructField("player_match_sk", IntegerType(), True),
    StructField("playermatch_key", DecimalType(), True),
    StructField("match_id", IntegerType(), True),
    StructField("player_id", IntegerType(), True),
    StructField("player_name", StringType(), True),
    StructField("dob", DateType(), True),
    StructField("batting_hand", StringType(), True),
    StructField("bowling_skill", StringType(), True),
    StructField("country_name", StringType(), True),
    StructField("role_desc", StringType(), True),
    StructField("player_team", StringType(), True),
    StructField("opposit_team", StringType(), True),
    StructField("season_year", IntegerType(), True),
    StructField("is_manofthematch", BooleanType(), True),
    StructField("age_as_on_match", IntegerType(), True),
    StructField("isplayers_team_won", BooleanType(), True),
    StructField("batting_status", StringType(), True),
    StructField("bowling_status", StringType(), True),
    StructField("player_captain", StringType(), True),
    StructField("opposit_captain", StringType(), True),
    StructField("player_keeper", StringType(), True),
    StructField("opposit_keeper", StringType(), True)
])



In [16]:
team_schema = StructType([
    StructField("team_sk", IntegerType(), True),
    StructField("team_id", IntegerType(), True),
    StructField("team_name", StringType(), True)
])




# Section 5 — Read CSV Files

In [17]:
# loading data from match.csv
ball_by_ball_df = spark.read.schema(ball_by_ball_schema).format("csv").option("header","true").load(Ball_By_Ball_file)

match_df = spark.read.schema(match_schema).format("csv").option("header","true").load(Match_file)

player_df = spark.read.schema(player_schema).format("csv").option("header","true").load(Player_file)

player_match_df = spark.read.schema(player_match_schema).format("csv").option("header","true").load(Player_match_file)

team_df = spark.read.schema(team_schema).format("csv").option("header","true").load(Team_file)

# Section 6 — Validation

In [18]:

print(type(ball_by_ball_schema))

<class 'pyspark.sql.types.StructType'>


In [19]:
# no of records count
print(
    ball_by_ball_df.count(),
    match_df.count(),
    player_df.count(),
    player_match_df.count(),
    team_df.count()
)

150451 637 497 13993 13


In [20]:
# validatios of df

ball_by_ball_df.printSchema()
ball_by_ball_df.show(5)

root
 |-- match_id: integer (nullable = true)
 |-- over_id: integer (nullable = true)
 |-- ball_id: integer (nullable = true)
 |-- innings_no: integer (nullable = true)
 |-- team_batting: string (nullable = true)
 |-- team_bowling: string (nullable = true)
 |-- striker_batting_position: integer (nullable = true)
 |-- extra_type: string (nullable = true)
 |-- runs_scored: integer (nullable = true)
 |-- extra_runs: integer (nullable = true)
 |-- wides: integer (nullable = true)
 |-- legbyes: integer (nullable = true)
 |-- byes: integer (nullable = true)
 |-- noballs: integer (nullable = true)
 |-- penalty: integer (nullable = true)
 |-- bowler_extras: integer (nullable = true)
 |-- out_type: string (nullable = true)
 |-- caught: boolean (nullable = true)
 |-- bowled: boolean (nullable = true)
 |-- run_out: boolean (nullable = true)
 |-- lbw: boolean (nullable = true)
 |-- retired_hurt: boolean (nullable = true)
 |-- stumped: boolean (nullable = true)
 |-- caught_and_bowled: boolean (null

26/06/26 06:58:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+-------+-------+----------+------------+------------+------------------------+----------+-----------+----------+-----+-------+----+-------+-------+-------------+--------------+------+------+-------+----+------------+-------+-----------------+----------+----------------+-------------+----------+------+-------+-----------+------+----------+--------+----------------+---------+-------------------+-------------+----------------+----------+---------------+---------+------------------+--------------+--------------+------------+-------------+-----------+
|match_id|over_id|ball_id|innings_no|team_batting|team_bowling|striker_batting_position|extra_type|runs_scored|extra_runs|wides|legbyes|byes|noballs|penalty|bowler_extras|      out_type|caught|bowled|run_out| lbw|retired_hurt|stumped|caught_and_bowled|hit_wicket|obstructingfeild|bowler_wicket|match_date|season|striker|non_striker|bowler|player_out|fielders|striker_match_sk|strikersk|nonstriker_match_sk|nonstriker_sk|fielder_match_sk|

In [21]:
# validatios of df

match_df.printSchema()
match_df.show(5)

root
 |-- match_sk: integer (nullable = true)
 |-- match_id: integer (nullable = true)
 |-- team1: string (nullable = true)
 |-- team2: string (nullable = true)
 |-- match_date: date (nullable = true)
 |-- season_year: integer (nullable = true)
 |-- venue_name: string (nullable = true)
 |-- city_name: string (nullable = true)
 |-- country_name: string (nullable = true)
 |-- toss_winner: string (nullable = true)
 |-- match_winner: string (nullable = true)
 |-- toss_name: string (nullable = true)
 |-- win_type: string (nullable = true)
 |-- outcome_type: string (nullable = true)
 |-- manofmach: string (nullable = true)
 |-- win_margin: integer (nullable = true)
 |-- country_id: integer (nullable = true)

+--------+--------+--------------------+--------------------+----------+-----------+--------------------+----------+------------+--------------------+--------------------+---------+--------+------------+-----------+----------+----------+
|match_sk|match_id|               team1|          

In [22]:
# validatios of df

player_df.printSchema()
player_df.show(5)

root
 |-- player_sk: integer (nullable = true)
 |-- player_id: integer (nullable = true)
 |-- player_name: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- batting_hand: string (nullable = true)
 |-- bowling_skill: string (nullable = true)
 |-- country_name: string (nullable = true)

+---------+---------+---------------+----+--------------+------------------+------------+
|player_sk|player_id|    player_name| dob|  batting_hand|     bowling_skill|country_name|
+---------+---------+---------------+----+--------------+------------------+------------+
|        0|        1|     SC Ganguly|NULL| Left-hand bat|  Right-arm medium|       India|
|        1|        2|    BB McCullum|NULL|Right-hand bat|  Right-arm medium| New Zealand|
|        2|        3|     RT Ponting|NULL|Right-hand bat|  Right-arm medium|   Australia|
|        3|        4|      DJ Hussey|NULL|Right-hand bat|Right-arm offbreak|   Australia|
|        4|        5|Mohammad Hafeez|NULL|Right-hand bat|Right-arm offb

In [23]:
# validatios of df
player_match_df.printSchema()
player_match_df.show(5)

root
 |-- player_match_sk: integer (nullable = true)
 |-- playermatch_key: decimal(10,0) (nullable = true)
 |-- match_id: integer (nullable = true)
 |-- player_id: integer (nullable = true)
 |-- player_name: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- batting_hand: string (nullable = true)
 |-- bowling_skill: string (nullable = true)
 |-- country_name: string (nullable = true)
 |-- role_desc: string (nullable = true)
 |-- player_team: string (nullable = true)
 |-- opposit_team: string (nullable = true)
 |-- season_year: integer (nullable = true)
 |-- is_manofthematch: boolean (nullable = true)
 |-- age_as_on_match: integer (nullable = true)
 |-- isplayers_team_won: boolean (nullable = true)
 |-- batting_status: string (nullable = true)
 |-- bowling_status: string (nullable = true)
 |-- player_captain: string (nullable = true)
 |-- opposit_captain: string (nullable = true)
 |-- player_keeper: string (nullable = true)
 |-- opposit_keeper: string (nullable = true)

+---

In [24]:
# validatios of df

team_df.printSchema()
team_df.show(5)

root
 |-- team_sk: integer (nullable = true)
 |-- team_id: integer (nullable = true)
 |-- team_name: string (nullable = true)

+-------+-------+--------------------+
|team_sk|team_id|           team_name|
+-------+-------+--------------------+
|      0|      1|Kolkata Knight Ri...|
|      1|      2|Royal Challengers...|
|      2|      3| Chennai Super Kings|
|      3|      4|     Kings XI Punjab|
|      4|      5|    Rajasthan Royals|
+-------+-------+--------------------+
only showing top 5 rows



In [25]:
ls

01_ipl_data_ingestion.ipynb
02_ipl_data_validate_write_bronze.ipynb*
03_ipl_data_Transform_silver.ipynb*
04_ipl_data_Transformation_gold.ipynb*
05_spark_sql_analytic.ipynb
